In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
# TO DO
import os
import glob
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np


In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
# TO DO
import os
import glob
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

# Custom Dataset Class for Flood Segmentation
class FloodSegmentationDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform


        self.img_paths = sorted(glob.glob(os.path.join(root_dir, "dataset","images", "*.*")))
        self.mask_paths = sorted(glob.glob(os.path.join(root_dir, "dataset","masks", "*.*")))



    def __len__(self):
        # Return the total number of samples in the dataset
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Get the specific paths for the current index
        img_path = self.img_paths[idx]
        mask_path = self.mask_paths[idx]

        # Load the image and convert to RGB
        image = Image.open(img_path).convert("RGB")

        # Load the mask and convert to Grayscale (L mode) for binary/class segmentation
        mask = Image.open(mask_path).convert("L")

        # Apply transformations to the image if provided
        if self.transform:
            image = self.transform(image)

        # Apply transformations to the mask if provided (e.g., Resize or ToTensor)
        if self.target_transform:
            mask = self.target_transform(mask)

        mask = remap_mask(mask)
        # For segmentation tasks, we return both the processed image and its corresponding mask
        return image, mask




In [ ]:

from torchvision import transforms
from torch.utils.data import DataLoader
from torch import nn

# Define transforms for images
img_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# need to subtract 1 from the make to bring it from 1-3 range to 0-2 range (CrossEntropyLoss requires labels to start from 0)
class SubtractOne(nn.Module):
  def forward(self, img):
    return img-1

# Define transforms for masks
target_transforms = transforms.Compose([                    ## Notice how we have a transform for the target (because it is an image)
    transforms.Resize((256, 256)),                          ##        and another one for the image itself.
    transforms.PILToTensor(),
    SubtractOne()                                           ## Question: What do you think would happen if we
                                                            ##  added rotation augmentation to the image only?
])


In [ ]:
# Load training dataset
# in segmentation we only call the sataset class once for the whole data
train_dataset = FloodSegmentationDataset(
                                        root_dir=(path),
                                         transform=img_transforms, target_transform=target_transforms)



In [ ]:
# Create Train & Test DataLoaders
from torch.utils.data import random_split, DataLoader, Subset
from torchvision import transforms


# Calculate split sizes
full_dataset_len = len(train_dataset)
train_size = int(0.8 * full_dataset_len)
test_size = full_dataset_len - train_size

# Generate random indices for splitting
train_indices, test_indices = random_split(range(full_dataset_len), [train_size, test_size])

# Create train and test datasets using Subset

train_dataset = Subset(train_dataset, train_indices)
test_dataset = Subset(train_dataset, test_indices)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [ ]:

images, labels = next(iter(train_loader))
print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")

In [ ]:
# Check dataset sizes
print(f"Training Samples: {len(train_dataset)}")
print(f"Training Samples: {len(test_loader)}")

In [ ]:
test_dataset[0]

In [ ]:
import matplotlib.pyplot as plt

# Display some images with their masks
for i in range(10,13):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.permute(1, 2, 0))  # Convert (C, H, W) to (H, W, C)
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()


In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Binary segmentation (1 output channel)
  # Apply Sigmoid activation directly in the model
).to(device)


In [ ]:
# TO DO
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long) # For any mulit class segmentation

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long) # For any mulit class segmentation

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss() # For any mulit class segmentation
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 2 # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO
import random
import matplotlib.pyplot as plt

model.eval()
# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass

    pred_mask = torch.softmax(pred_mask, dim=1)  # Convert logits to probabilities
    pred_mask = pred_mask.argmax(dim=1).cpu().squeeze().numpy()  # Get class with highest probability

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")  # Show class map
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
